In [1]:
##### Creates rasters of HA of ag land area (crop + livestock)
### Combines MapSPAM physical area across all crops with grazeland data
# unit = ha

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from pathlib import Path
from rasterio.warp import reproject
from rasterio.enums import Resampling
from rasterio.windows import Window, from_bounds

In [2]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# reference raster
ref_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# grassland raster
raw_grassland = f"{cd}/Data/Raw/Predictors/Grazeland_Parente/gpw_grassland_rf.med.filt.bthr.reg_c_30m_20200101_20201231_go_epsg.4326_v2.tif"

In [3]:
##### Add MapSPAM values to get total HA of crop production 

# set paths
spam_dir = Path(f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_physical_area/")
output_path = f"{cd}/Data/Raw/Predictors/Ag_land_ME/crop_land_ha_2020.tif"

# get all technology rasters 
a_rasters = sorted(spam_dir.glob("spam*_A.tif"))

with rasterio.open(a_rasters[0]) as src:
    meta  = src.meta.copy()
    shape = src.shape

total = np.zeros(shape, dtype=np.float64)

for path in a_rasters:
    with rasterio.open(path) as src:
        data = src.read(1).astype(np.float64)
        data = np.where(np.isnan(data), 0, data)  # nan → 0 before summing
        total += data

meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")
out_arr = total.astype(np.float32)
out_arr[out_arr == 0] = -9999  # pixels with no cropland → nodata

with rasterio.open(output_path, "w", **meta) as dst:
    dst.write(out_arr, 1)